#  LangGraph Part B: Theory Notes
## Loops (Cycles), Memory & Human-in-the-Loop



| Topic | Concept |
|-------|---------|
| 1 | Loops / Cycles in StateGraph |
| 2 | Message Accumulation with `operator.add` |
| 3 | Memory & Checkpoints with `MemorySaver` |
| 4 | Human-in-the-Loop (HITL) with `interrupt_before` |

---
## 1. Loops (Cycles) in LangGraph

### What is a Cycle?
In a normal pipeline graph, execution flows in **one direction** only:
```
START → node_A → node_B → END
```
A **cycle** means one node has an edge pointing **back** to an earlier node, allowing the graph to loop.

```
START → [agent] ──── has tool_calls? ────→ YES → [tools] ──┐
              └──── NO → END                                └──→ back to [agent]
```

---

### Why are Cycles Needed?

| Scenario | Why cycle is needed |
|----------|--------------------|
| Agent ↔ Tools loop | LLM may need multiple tool calls before it has enough info to answer |
| Iterative refinement | A node keeps improving output until a quality threshold is met |
| Retry logic | Re-run a node if it fails or returns an invalid result |
| Multi-step reasoning | Chain-of-thought that requires going back and forth |

---

### How to Create a Cycle
Simply add an edge from a **later node back to an earlier node**:

```python
agent_builder.add_edge(START,   "agent")   # entry
agent_builder.add_edge("tools", "agent")   # ← THIS is the back-edge (cycle)
agent_builder.add_conditional_edges("agent", should_continue)
# should_continue returns "tools" or END
```

---

### Agent ↔ Tools Cycle — Step-by-Step Flow

```
Step 1:  User sends query  →  HumanMessage added to state

Step 2:  [agent node] runs
         → LLM sees all messages in state
         → Decides: do I need a tool? YES
         → Returns AIMessage with tool_calls=[...]
         → This AIMessage is APPENDED to state

Step 3:  should_continue() checks last message
         → last message has tool_calls → return "tools"

Step 4:  [tools node] runs
         → Executes each tool call
         → Returns ToolMessage(s) with results
         → ToolMessages are APPENDED to state

Step 5:  Back to [agent node]  (cycle!)
         → LLM sees full history including tool results
         → Decides: I have enough info, give final answer
         → Returns AIMessage with plain text answer
         → This AIMessage is APPENDED to state

Step 6:  should_continue() checks last message
         → last message has NO tool_calls → return END

Step 7:  Graph exits. Final answer is state['messages'][-1]
```

---

### Routing Function — The Brain of the Cycle

```python
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]           # get the freshest message
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"   # loop back
    return END           # exit the graph
```

| Condition | `last.tool_calls` | Decision |
|-----------|------------------|----------|
| LLM wants to call a tool | Not empty list | → go to `tools` node (loop) |
| LLM has a final answer | Empty / None | → go to `END` (stop) |

---

### Single-Tool vs Multi-Tool Query

**Simple query:** `"What is 47 + 89?"`
```
Messages in state:
1. HumanMessage  → "What is 47 + 89?"
2. AIMessage     → tool_calls=[add_numbers(47, 89)]
3. ToolMessage   → "136"
4. AIMessage     → "47 plus 89 is 136."
Total: 4 messages, 1 cycle
```

**Multi-tool query:** `"What is today's date AND sqrt of 225?"`
```
Messages in state:
1. HumanMessage  → "What is today's date AND sqrt of 225?"
2. AIMessage     → tool_calls=[get_current_date(), calculate_square_root(225)]
3. ToolMessage   → "2026-03-12"       (result of get_current_date)
4. ToolMessage   → "15.0"             (result of calculate_square_root)
5. AIMessage     → "Today is 2026-03-12 and sqrt(225) = 15."
Total: 5 messages, 1 cycle (both tools called in same cycle)
```

---
## 2. Message Accumulation with `operator.add`

### The Problem Without `operator.add`
Normally, when a node returns `{"messages": [...]}`, LangGraph **replaces** the existing `messages` in state with the new list.

This is **catastrophic for cycles** because the LLM would lose all conversation history on every loop iteration.

### The Solution — `Annotated` with `operator.add`

```python
from typing import Annotated
import operator

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
#                                          ^^^^^^^^^^^^^
#                       This is a REDUCER — tells LangGraph HOW to merge updates
```

| Without `operator.add` | With `operator.add` |
|------------------------|--------------------|
| Node returns `[msg3]` → state becomes `[msg3]` (overwrites!) | Node returns `[msg3]` → state becomes `[msg1, msg2, msg3]` (appends!) |
| History is LOST each cycle | History ACCUMULATES across cycles |

---

### How `operator.add` Works

`operator.add` is just Python's `+` operator for lists:
```python
import operator
operator.add([1, 2, 3], [4, 5])   # → [1, 2, 3, 4, 5]
```

LangGraph uses it as a **reducer function**:
```
new_state["messages"] = operator.add(old_state["messages"], node_return["messages"])
                      = old_messages + new_messages
                      = merged/appended list
```

---

### Why `state["messages"][-1]` Always Gets the Latest Message

Because every new message is **appended** to the end of the list, the most recent message is always at index `-1`.

```
After cycle 1:  messages = [HumanMsg, AIMsg(tool_call)]   → [-1] = AIMsg(tool_call)
After tools:    messages = [HumanMsg, AIMsg, ToolMsg]      → [-1] = ToolMsg
After cycle 2:  messages = [HumanMsg, AIMsg, ToolMsg, AIMsg(answer)] → [-1] = AIMsg(answer)
```

---

### Message Types in the Conversation

| Message Type | Class | Created by | Contains |
|-------------|-------|------------|----------|
| User input | `HumanMessage` | Your code | The user's question |
| LLM response (wants tool) | `AIMessage` | LLM | `tool_calls=[...]` list |
| LLM response (final answer) | `AIMessage` | LLM | Plain text content |
| Tool result | `ToolMessage` | `tools_exec_node` | Tool output + `tool_call_id` |

The **`tool_call_id`** in `ToolMessage` must match the ID in the `AIMessage`'s `tool_calls` — this links the result back to the request.

---
## 3. Memory & Checkpoints with `MemorySaver`

### The Problem — Stateless by Default
Without a checkpointer, every `.invoke()` call starts **completely fresh**:

```python
graph.invoke({"messages": [HumanMessage("My name is Alice.")]})
graph.invoke({"messages": [HumanMessage("What is my name?")]})  # LLM says "I don't know"
```

Each call is isolated — the graph has **no memory** of the previous call.

---

### The Solution — Checkpointer + `thread_id`

```python
from langgraph.checkpoint.memory import MemorySaver

mem_saver = MemorySaver()                           # in-memory store
graph = builder.compile(checkpointer=mem_saver)     # attach to graph

config = {"configurable": {"thread_id": "alice-001"}}
graph.invoke({...}, config=config)                  # pass config every call
```

---

### How Checkpointing Works Internally

```
Call 1:  invoke(HumanMessage("My name is Alice."), config={thread_id: "alice-001"})
         → Graph runs
         → After completion, full state is SAVED to MemorySaver under key "alice-001"
         → State = [HumanMsg("My name is Alice."), AIMsg("Hello Alice!")]

Call 2:  invoke(HumanMessage("I work at a fintech."), config={thread_id: "alice-001"})
         → Graph LOADS saved state for "alice-001"
         → APPENDS new HumanMessage to loaded state (operator.add)
         → LLM sees full history: Alice's name + new message
         → After completion, saves updated state back

Call 3:  invoke(HumanMessage("What do you know about me?"), config={thread_id: "alice-001"})
         → Loads full history (3 human + 2 AI messages)
         → LLM recalls everything → "You are Alice, and you work at a fintech."
```

---

### `thread_id` — The Session Key

The `thread_id` acts like a **session identifier**.

| `thread_id` | Result |
|------------|--------|
| Same ID across calls | Same state is restored → graph **remembers** |
| Different ID | No saved state found → **fresh start**, no memory |

```
alice-001 thread:          bob-001 thread:
─────────────────          ────────────────
Turn 1: "I'm Alice"        Turn 1: "What is my name?"
Turn 2: "I use AWS"        AI: "I don't know your name" ← Bob starts fresh!
Turn 3: "What do you       (Bob's session has NO knowledge
         know about me?"    of Alice's session)
AI: "You are Alice, 
     a cloud architect 
     using AWS"
```

---

### Checkpointer Types

| Checkpointer | Storage | Use case |
|-------------|---------|----------|
| `MemorySaver` | RAM (in-process) | Development, testing, notebooks |
| `SqliteSaver` | SQLite file on disk | Single-server production |
| `PostgresSaver` | PostgreSQL database | Scalable production, multi-server |

> `MemorySaver` is lost when the process exits. For persistent memory across restarts, use `SqliteSaver` or `PostgresSaver`.

---

### State Growth Over Turns

Since `operator.add` appends, the state list **grows** with every turn:

```
After Turn 1:  [HumanMsg_1, AIMsg_1]
After Turn 2:  [HumanMsg_1, AIMsg_1, HumanMsg_2, AIMsg_2]
After Turn 3:  [HumanMsg_1, AIMsg_1, HumanMsg_2, AIMsg_2, HumanMsg_3, AIMsg_3]
```

This growing history is what **gives the LLM context** about previous turns — the entire conversation is sent to Gemini on every call.

---
## 4. Human-in-the-Loop (HITL) with `interrupt_before`

### What is HITL?
HITL pauses a running graph at a specific node so a **human can review, approve, or reject** the proposed action before it executes.

**Real-world use cases:**
- Approve/reject a database write before it happens
- Review an email draft before sending
- Confirm a financial transaction before processing
- Validate an AI-generated API call before execution

---

### Setup — Two Requirements

HITL needs **both** of these:

```python
hitl_ckpt  = MemorySaver()              # 1. Checkpointer — to save the paused state
hitl_graph = agent_builder.compile(
    checkpointer=hitl_ckpt,
    interrupt_before=["tools"],         # 2. interrupt_before — which node to pause before
)
```

| Requirement | Why it's needed |
|------------|----------------|
| `checkpointer` | The paused state must be **stored** so execution can resume later |
| `interrupt_before` | Tells LangGraph **which node** to pause before |

---

### HITL Flow — Full Picture

```
─────────────────────────────────────────────────────────────────────
INVOKE 1  →  Graph starts running
          →  [agent node] runs: LLM proposes tool call
          →  LangGraph sees: next node is "tools" which is in interrupt_before
          →  PAUSE. State saved to checkpointer. Returns paused state.
─────────────────────────────────────────────────────────────────────
              ↓      Human reviews paused['messages'][-1].tool_calls
─────────────────────────────────────────────────────────────────────
APPROVED  →  INVOKE 2 (invoke(None, config))
          →  Graph loads saved state from checkpointer
          →  Resumes from pause point
          →  [tools node] runs: executes the tool
          →  [agent node] runs: LLM gives final answer
          →  Graph exits normally
─────────────────────────────────────────────────────────────────────
REJECTED  →  Don't call invoke(None, ...) at all
          →  Graph stays frozen in checkpointer
          →  Tool is NEVER executed
─────────────────────────────────────────────────────────────────────
```

---

### The `invoke(None, config)` Call — What Does `None` Mean?

```python
# Normal invocation — you provide new input
graph.invoke({"messages": [HumanMessage("...")]}, config=cfg)

# Resume invocation — None means "no new input, just continue"
graph.invoke(None, config=cfg)
```

`None` tells LangGraph:
> "I'm not starting a new run. Load the paused state for this `thread_id` and continue from where it was interrupted."

---

### Approval vs Rejection — Summary

| Decision | Code | Effect |
|----------|------|--------|
| **APPROVE** | `graph.invoke(None, config=cfg)` | Resumes graph → tool runs → final answer produced |
| **REJECT** | *(don't call invoke)* | Graph stays paused → tool never runs |

There is **no explicit "reject" function** — rejection is simply the absence of resumption.

---

### Inspecting the Paused State

After the first `invoke()` returns, you can inspect what the LLM proposed:

```python
paused = hitl_graph.invoke({"messages": [HumanMessage(query)]}, config=cfg)

last_msg = paused["messages"][-1]         # AIMessage from LLM

if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"Tool : {tc['name']}")      # e.g. calculate_square_root
        print(f"Args : {tc['args']}")      # e.g. {'number': 625}
```

The human can verify:
- **Which tool** is about to run
- **What arguments** will be passed
- Whether this looks **safe and correct** before approving

---

### Normal Graph vs HITL Graph — Comparison

| Aspect | Normal Graph | HITL Graph |
|--------|-------------|------------|
| Execution | Fully automatic | Pauses before specified node |
| Checkpointer | Optional | **Required** |
| `invoke()` calls | 1 call per query | 2 calls (start + resume) for approved |
| Human involvement | None | Human reviews between the 2 calls |
| Tool execution | Always happens | Only if human approves |
| Use case | Trusted, automated tasks | Sensitive, high-risk, or irreversible actions |

---

### Advanced: Modifying State Before Resuming

Before calling `invoke(None, ...)`, you can **modify the state** in the checkpointer. This allows you to:
- Change the tool arguments before execution
- Inject a rejection message so the LLM understands why the tool wasn't run
- Override the LLM's decision entirely

```python
# Get and modify state before resuming
current_state = hitl_graph.get_state(config=cfg)
# ... modify current_state.values ...
hitl_graph.update_state(config=cfg, values=modified_state)
# Then resume with modified state
hitl_graph.invoke(None, config=cfg)
```

---
## 5. How All Three Concepts Work Together

Loops, Memory, and HITL are **not separate features** — they build on each other:

```
LOOPS
  └── Require operator.add so history accumulates across cycle iterations

MEMORY
  └── Uses MemorySaver to persist state across .invoke() calls
  └── Also uses operator.add so new messages are appended to saved history

HITL
  └── Requires MemorySaver to save the paused state between the two invoke() calls
  └── Uses the same cyclic agent graph (just compiled with interrupt_before)
  └── Uses operator.add so state is preserved through the pause
```

### Dependency Chain
```
operator.add  →  enables LOOPS  →  enables MEMORY  →  enables HITL
```

---

## 6. Quick Reference — Key APIs

| Concept | API | Purpose |
|---------|-----|---------|
| Message accumulation | `Annotated[list[BaseMessage], operator.add]` | Append instead of overwrite |
| Create back-edge (cycle) | `builder.add_edge("tools", "agent")` | Send tools output back to agent |
| Routing function | `add_conditional_edges("agent", should_continue)` | Loop or stop |
| Get latest message | `state["messages"][-1]` | Always returns freshest message |
| In-memory store | `MemorySaver()` | Save/restore state between calls |
| Attach to graph | `builder.compile(checkpointer=mem)` | Enable persistence |
| Set session | `config={"configurable": {"thread_id": "..."}}` | Identify which thread to restore |
| HITL pause point | `compile(interrupt_before=["node_name"])` | Where to freeze execution |
| HITL approve | `graph.invoke(None, config=cfg)` | Resume from saved checkpoint |
| HITL reject | *(don't call invoke)* | Leave graph frozen |
| Inspect paused state | `paused["messages"][-1].tool_calls` | See proposed tool call |